# August 2026 verification · **04 — Reconciliation gate**

**Nothing downstream runs until this passes.** The last cell raises if any
check fails, so `jupyter nbconvert --execute` exits non-zero, and a failed
gate cannot be missed in a run log.

I rebuild NHS England's published August figures from my own Silver tables,
on two measures, and read every target out of NHS England's own files (the
workbooks directly, and the PDFs by text extraction). Nothing is typed in from
the BMA.

| gate | measure | what must hold |
|---|---|---|
| **A** | reported-provider, NONC excluded (**primary**) | Aug26 and Jul26 provider workbooks, every provider, exact; every month Apr25-Aug26 against the reported columns of the Overview series; July = Two Maps' 7,131,685 / 108,263 |
| **B** | headline = reported + NHS England's missing-trust estimates | every month, exact, against the "with estimates" columns; the press notice's 113,339 |
| **M** | median / 92nd percentile | rebuilt from the weekly bands, matching the press notice to its one decimal place |
| **P** | unique patients | **sourced, not rebuilt.** The WLMDS ratio is not in any provider file. I check only that NHS England's own arithmetic holds |
| **C** | Cancer Waiting Times, provider basis | the August and July release percentages, and the national time series counts, exact |

After the gate, a second table puts each secondary-source claim beside the
primary figure and says how it relates.

In [1]:
import re
import sys
from pathlib import Path

import numpy as np
import openpyxl
import pandas as pd
import pypdf

sys.path.insert(0, str(Path.cwd().parent / "src"))
from _common import (BRONZE_DIR, GOLD_DIR, SILVER_DIR,  # noqa: E402
                     TWO_MAPS_JUL26_OVER_52, TWO_MAPS_JUL26_TOTAL)
from claims import CLAIMS  # noqa: E402

OUT = GOLD_DIR / "00_reconciliation"
OUT.mkdir(parents=True, exist_ok=True)
PUB = BRONZE_DIR / "nhse_rtt_published_2026-08"
CWT = BRONZE_DIR / "nhse_cwt_2026-08"
pd.set_option("display.width", 240)
pd.set_option("display.max_colwidth", 90)

rtt = pd.read_parquet(SILVER_DIR / "rtt" / "rtt_provider.parquet")
bands = pd.read_parquet(SILVER_DIR / "rtt" / "rtt_england_bands.parquet")
cwt = pd.read_parquet(SILVER_DIR / "cwt" / "cwt_provider_headline.parquet")
inc = rtt[rtt.part == "incomplete"]

checks = []
def check(gate, name, published, rebuilt, ok, note=""):
    checks.append({"gate": gate, "check": name, "published": published, "rebuilt": rebuilt,
                   "diff": (rebuilt - published) if isinstance(published, (int, float, np.number)) and
                            isinstance(rebuilt, (int, float, np.number)) else None,
                   "pass": bool(ok), "note": note})

## Reading NHS England's targets

### The press notices (PDF → text → regex)

I pull each figure out with a pattern anchored on NHS England's own wording.
If a pattern doesn't match, the notebook stops. A missing target must never
turn into a silent pass.

In [2]:
def pdf_text(path: Path) -> str:
    return re.sub(r"\s+", " ", " ".join(p.extract_text() for p in pypdf.PdfReader(path).pages))

def grab(text: str, pattern: str, cast=float):
    m = re.search(pattern, text)
    assert m, f"pattern not found in the press notice: {pattern}"
    return cast(m.group(1).replace(",", ""))

spn = pdf_text(PUB / "Aug26-RTT-statistical-press-notice-PDF-389K-vjgRdG.pdf")
SPN = {
    "pathways_rounded_m": grab(spn, r"end of August 2026 was ([\d.]+) million"),
    "patients_rounded_m": grab(spn, r"unique patients is estimated to be around ([\d.]+) million"),
    "over_52_est": grab(spn, r"in ([\d,]+) cases the patient was waiting more than 52 weeks", int),
    "pct_within_18_est": grab(spn, r"In ([\d.]+)% of cases the patient had been waiting up to 18 weeks"),
    "median_weeks": grab(spn, r"the median waiting time was ([\d.]+) weeks"),
    "p92_weeks": grab(spn, r"92nd percentile waiting time was ([\d.]+) weeks"),
    "table1_total_reported": grab(spn, r"Other - Other Services [\d,]+ [\d.]+% [\d,]+ [\d,]+ [\d,]+ Total ([\d,]+) ", int),
    "wlmds_ratio": grab(spn, r"unique NHS numbers to pathways on 30th August 2026 was (\d+):100") / 100,
    "yoy_change": grab(spn, r"decreased by [\d.]+% \(([\d,]+)\) compared to the end of August 2025", int),
}
SPN

{'pathways_rounded_m': 7.4,
 'patients_rounded_m': 6.2,
 'over_52_est': 113339,
 'pct_within_18_est': 65.2,
 'median_weeks': 12.2,
 'p92_weeks': 38.8,
 'table1_total_reported': 7158164,
 'wlmds_ratio': 0.85,
 'yoy_change': 33512}

In [3]:
rel = pdf_text(CWT / "Cancer-Waiting-Times-Statistical-Release-August-2026-Provider-based-Provisional.pdf")
REL = {
    "FDS_2026-08": grab(rel, r"([\d.]+)% of people were told if they had cancer"),
    "FDS_2026-07": grab(rel, r"definitively excluded within four weeks \(28-days\) of an urgent referral \(([\d.]+)% in July 2026\)"),
    "31D_2026-08": grab(rel, r"([\d.]+)% of people treated began first or subsequent treatment"),
    "31D_2026-07": grab(rel, r"earliest clinically appropriate date \(([\d.]+)% in July 2026\)"),
    "62D_2026-08": grab(rel, r"([\d.]+)% of people treated began first definitive treatment"),
    "62D_2026-07": grab(rel, r"or a consultant upgrade \(([\d.]+)% in July 2026\)"),
    "FDS_total_2026-08": grab(rel, r"([\d,]+) referrals on the Faster Diagnosis Standard \(FDS\) were seen in August 2026", int),
    "62D_total_2026-08": grab(rel, r"([\d,]+) referrals on the 62D combined standard \(62D\) were seen in August 2026", int),
}
REL

{'FDS_2026-08': 78.6,
 'FDS_2026-07': 79.3,
 '31D_2026-08': 92.1,
 '31D_2026-07': 92.5,
 '62D_2026-08': 71.9,
 '62D_2026-07': 71.2,
 'FDS_total_2026-08': 273664,
 '62D_total_2026-08': 27909}

### The Overview time series (headline series, both measures side by side)

Columns are located by their header text, not position. Despite the header
"(mil)", the "Total waiting" columns hold whole pathway counts. I assert that
rather than assume it.

In [4]:
ws = openpyxl.load_workbook(PUB / "RTT-Overview-Timeseries-Including-Estimates-for-Missing-Trusts-Aug26-XLS-139K-vjgRdG.xlsx",
                            read_only=True, data_only=True).worksheets[0]
rows = list(ws.iter_rows(values_only=True))
hdr = [str(h).strip() if h else "" for h in rows[11]]
def col(label, nth=0):
    hits = [i for i, h in enumerate(hdr) if h.startswith(label)]
    return hits[nth]
ov_cols = {
    "total_rep": col("Total waiting (mil)"), "total_est": col("Total waiting (mil) with estimates"),
    "over_18_rep": col("No. > 18 weeks"), "over_18_est": col("No. > 18 weeks with estimates"),
    "over_52_rep": col("No. > 52 weeks"), "over_52_est": col("No. > 52 weeks with estimates"),
    "within_18_rep": col("No. within 18 weeks"), "within_18_est": col("No. within 18 weeks with estimates"),
    "median": col("Median wait (weeks)"), "p92": col("92nd percentile (weeks)"),
    "ratio": col("Ratio of total unique patients"), "patients": col("Estimated number of unique patients"),
}
# 'No. > 18 weeks' also prefixes its 'with estimates' twin; the first hit is the plain one.
assert "estimates" not in hdr[ov_cols["over_52_rep"]] and "estimates" not in hdr[ov_cols["total_rep"]]
ov = pd.DataFrame([{k: r[i] for k, i in ov_cols.items()} | {"period": r[2].strftime("%Y-%m")}
                   for r in rows[12:] if hasattr(r[2], "strftime") and r[ov_cols["total_rep"]] not in (None, "-")])
ov = ov[(ov.period >= "2025-04") & (ov.period <= "2026-08")].set_index("period")
assert (ov.total_rep > 1e6).all(), "Total waiting is not a whole count after all"
ov[["total_rep", "total_est", "over_52_rep", "over_52_est", "ratio", "patients"]].tail(3)

,total_rep,total_est,over_52_rep,over_52_est,ratio,patients
period,,,,,,
2026-06,7147562,7272291,103318,105711,0.846042,6152666.731512
2026-07,7131685,7328252,108263,111379,0.847006,6207072.785461
2026-08,7158164,7370100,109936,113339,0.846972,6242270.833899


## Gate A — reported-provider, provider by provider

The Incomplete Provider workbook splits NHS and independent-sector providers
across two sheets. Together they are the whole reported list. I match on
provider code and require every provider to agree on total, within 18 weeks,
over 52 weeks and 104+. A national total can agree by accident; 533
providers agreeing at once can't.

In [5]:
def workbook_providers(path: Path, dta: bool = False) -> pd.DataFrame:
    sheets = ["Provider with DTA", "IS Provider with DTA"] if dta else ["Provider", "IS Provider"]
    out = []
    for sh in sheets:
        d = pd.read_excel(path, sheet_name=sh, header=13)
        d = d[d["Treatment Function Code"] == "C_999"]
        tot = [c for c in d.columns if str(c).startswith("Total number of incomplete pathways")][0]
        keep = {"Provider Code": "provider_code", tot: "total", "104 plus": "over_104"}
        if not dta:
            keep |= {"Total within 18 weeks": "within_18", "Total 52 plus weeks": "over_52",
                     "Average (median) waiting time (in weeks)": "median_pub"}
        out.append(d[list(keep)].rename(columns=keep).assign(sheet=sh))
    w = pd.concat(out, ignore_index=True)
    assert w.provider_code.is_unique
    return w

def gate_a_month(period: str, path: Path):
    for dta in (False, True):
        part = "incomplete_dta" if dta else "incomplete"
        wb = workbook_providers(path, dta)
        mine = rtt[(rtt.period == period) & (rtt.part == part)]
        m = wb.merge(mine, on="provider_code", how="outer", suffixes=("_pub", ""), indicator=True)
        # The workbook lists providers with zero pathways (the DTA sheets list every
        # provider; ACES Bolton appears in August with 0 incomplete pathways because
        # it only reported a completed one). The CSV extract has no row for them. A
        # workbook-only provider is acceptable ONLY if its published total is exactly
        # zero; a provider in my Silver that the workbook lacks is always a failure.
        wb_only = m[m._merge == "left_only"]
        mine_only = m[m._merge == "right_only"]
        zero_listed = wb_only.total_pub.eq(0).all()
        only = m.iloc[0:0] if (zero_listed and mine_only.empty) else m[m._merge != "both"]
        check("A", f"{period} {part}: same provider set (zero-pathway listings allowed)", len(wb), len(mine), only.empty,
              f"{len(wb_only)} workbook-only providers, all with 0 pathways; none missing from the workbook"
              if only.empty else f"unmatched with pathways: {only.provider_code.tolist()[:10]}")
        for c in (["total", "within_18", "over_52", "over_104"] if not dta else ["total", "over_104"]):
            bad = m[(m._merge == "both") & (m[f"{c}_pub"] != m[c])]
            check("A", f"{period} {part}: {c}, provider by provider", int(wb[c].sum()), int(mine[c].sum()),
                  bad.empty and only.empty, f"{len(bad)} providers differ" if len(bad) else "all providers exact")
    return workbook_providers(path)

wb_aug = gate_a_month("2026-08", PUB / "Incomplete-Provider-Aug26-XLSX-9M-vjgRdG.xlsx")
wb_jul = gate_a_month("2026-07", PUB / "Incomplete-Provider-Jul26-XLSX-9M-97ylx5.xlsx")

### A′ — the same measure as Two Maps

In [6]:
j = inc[inc.period == "2026-07"]
check("A'", "Jul26 total = Two Maps", TWO_MAPS_JUL26_TOTAL, int(j.total.sum()), int(j.total.sum()) == TWO_MAPS_JUL26_TOTAL,
      "proves this pipeline uses the Two Maps measure")
check("A'", "Jul26 over 52 = Two Maps", TWO_MAPS_JUL26_OVER_52, int(j.over_52.sum()), int(j.over_52.sum()) == TWO_MAPS_JUL26_OVER_52)
a = inc[inc.period == "2026-08"]
check("A", "Aug26 total = press notice Table 1 (not incl. estimates)", SPN["table1_total_reported"], int(a.total.sum()),
      int(a.total.sum()) == SPN["table1_total_reported"])

### A, every month — the index run must sit on the same measure throughout

In [7]:
rep = inc.groupby("period")[["total", "within_18", "over_18", "over_52"]].sum()
for c in ["total", "within_18", "over_18", "over_52"]:
    pub = ov[f"{c}_rep"].astype("int64")
    bad = (rep[c] != pub)
    check("A", f"Apr25-Aug26 {c} vs Overview reported column, each month", int(pub.sum()), int(rep[c].sum()),
          not bad.any(), "all 17 months exact" if not bad.any() else f"differs in {list(bad[bad].index)}")

## Gate B — rebuilding the headline (including estimates)

NHS England's method, as its press notice states it: for a trust that didn't
submit, carry forward its **last submitted** incomplete-pathway position. The
*Missing data adjustments* file lists, per month, which trust was estimated,
from which base month, and the pathways it added. I rebuild the headline as:

* **total**: reported + the file's estimated pathways;
* **over 52 / over 18 weeks**: reported + each missing trust's over-52 (over-18) count **in its base month, from my own Silver**.

The second is an independent test. NHS England doesn't publish the
estimated over-52 counts per trust. If my carry-forward lands exactly on its
"with estimates" column, I have reproduced the method, not just the number.

In [8]:
adj = pd.read_excel(PUB / "Missing-data-adjustments-to-Aug26-XLSX-109K-vjgRdG.xlsx", sheet_name="Adjustments")
adj = adj[adj["RTT Type"].eq("Incomplete")].copy()
adj["period"] = adj["Estimated month"].dt.strftime("%Y-%m")
# NHS England dates base months either as the 1st or as the last day; both mean that month.
adj["base"] = adj["Base month"].dt.strftime("%Y-%m")
adj = adj[(adj.period >= "2025-04") & (adj.period <= "2026-08")]
print(adj.groupby("period").apply(lambda g: ", ".join(f"{c} (base {b})" for c, b in zip(g["Prov Code"], g.base)),
                                  include_groups=False).to_string())

base = inc.set_index(["period", "provider_code"])
est = adj.assign(
    base_total=[base.total.get((b, c), np.nan) for b, c in zip(adj.base, adj["Prov Code"])],
    base_over_52=[base.over_52.get((b, c), np.nan) for b, c in zip(adj.base, adj["Prov Code"])],
    base_over_18=[base.over_18.get((b, c), np.nan) for b, c in zip(adj.base, adj["Prov Code"])],
)
# The estimate file's own pathway figure must be the base month's total in my Silver.
check("B", "each estimate = that trust's base-month total in my Silver", int(est["Estimated RTT Pathways"].sum()),
      int(est.base_total.sum()), (est["Estimated RTT Pathways"] == est.base_total).all(),
      f"{len(est)} trust-months estimated")

est_m = est.groupby("period")[["Estimated RTT Pathways", "base_over_52", "base_over_18"]].sum()
head = rep.join(est_m, how="left").fillna(0)
head["total_est"] = head.total + head["Estimated RTT Pathways"]
head["over_52_est"] = head.over_52 + head.base_over_52
head["over_18_est"] = head.over_18 + head.base_over_18
for c in ["total_est", "over_52_est", "over_18_est"]:
    pub = ov[c].astype("int64")
    bad = head[c].astype("int64") != pub
    check("B", f"Apr25-Aug26 {c} = reported + carry-forward, each month", int(pub.sum()), int(head[c].sum()),
          not bad.any(), "all 17 months exact" if not bad.any() else f"differs in {list(bad[bad].index)}")

# I save the rebuilt headline so Gold reads the series this gate proved, not a
# second copy of the method.
head[["total", "within_18", "over_18", "over_52", "total_est", "over_18_est", "over_52_est"]].astype("int64") \
    .to_csv(OUT / "headline_rebuilt.csv")

h = head.loc["2026-08"]
check("B", "Aug26 over 52 (incl. estimates) = press notice", SPN["over_52_est"], int(h.over_52_est), int(h.over_52_est) == SPN["over_52_est"])
check("B", "Aug26 pathways (incl. estimates), to the notice's 0.1m", SPN["pathways_rounded_m"], round(h.total_est / 1e6, 1),
      round(h.total_est / 1e6, 1) == SPN["pathways_rounded_m"])
pct18 = 100 * (h.total_est - h.over_18_est) / h.total_est
check("B", "Aug26 % within 18 weeks (incl. estimates)", SPN["pct_within_18_est"], round(pct18, 1), round(pct18, 1) == SPN["pct_within_18_est"])
yoy = int(head.loc["2025-08", "total_est"] - h.total_est)
check("B", "Aug26 vs Aug25 fall in pathways (incl. estimates)", SPN["yoy_change"], yoy, yoy == SPN["yoy_change"])

period
2025-07                                                                RHQ (base 2025-06)
2025-08                                                                RHQ (base 2025-06)
2025-09                                                                RHQ (base 2025-06)
2025-10                                                                RHQ (base 2025-06)
2025-11                                            RF4 (base 2025-10), RHQ (base 2025-06)
2025-12                                                                RHQ (base 2025-06)
2026-01                                                                RHQ (base 2025-06)
2026-02                                                                RHQ (base 2025-06)
2026-03                                                                RHQ (base 2025-06)
2026-04                                            RA9 (base 2026-03), RHQ (base 2025-06)
2026-05                                            RA9 (base 2026-03), RHQ (base 2025-06)
202

## Gate M — median and 92nd percentile from the weekly bands

The press notice says these are calculated "from aggregate data" and exclude
estimates. So I rebuild them from the reported England band distribution, by
linear interpolation within the one-week band where the 50th (92nd)
percentile falls. I test the method twice: nationally against the notice, and
provider by provider against the workbook's own medians.

In [9]:
def percentile_weeks(counts: np.ndarray, q: float) -> float:
    cum = np.cumsum(counts)
    target = q * cum[-1]
    k = int(np.searchsorted(cum, target))
    before = cum[k - 1] if k else 0
    return k + (target - before) / counts[k]

eb = bands[bands.part == "incomplete"].sort_values(["period", "week_lower"])
med = eb.groupby("period").pathways.apply(lambda s: percentile_weeks(s.to_numpy(), 0.5))
p92 = eb.groupby("period").pathways.apply(lambda s: percentile_weeks(s.to_numpy(), 0.92))
check("M", "Aug26 median weeks", SPN["median_weeks"], round(med["2026-08"], 1), round(med["2026-08"], 1) == SPN["median_weeks"])
check("M", "Aug26 92nd percentile weeks", SPN["p92_weeks"], round(p92["2026-08"], 1), round(p92["2026-08"], 1) == SPN["p92_weeks"])
mbad = (med.round(2) - ov["median"].astype(float).round(2)).abs()
check("M", "Apr25-Aug26 median vs Overview series, each month (2 d.p.)", round(float(ov["median"].astype(float).mean()), 2),
      round(float(med.mean()), 2), (mbad < 0.011).all(), f"max |diff| {mbad.max():.3f} weeks")

## Gate P — unique patients: sourced, not rebuilt

This is where the pathways-vs-patients claim actually comes from. NHS
England's "unique patients" is **not a count**. It is a ratio of unique NHS
numbers to pathways in the WLMDS (a separate, patient-level collection, taken
on 30 August 2026), multiplied by the monthly pathway total including
estimates. No provider file contains it. All I can check is that NHS
England's arithmetic is internally consistent.

In [10]:
r_ = float(ov.loc["2026-08", "ratio"]); pts = float(ov.loc["2026-08", "patients"])
check("P", "Overview ratio, to 2 d.p. = the notice's 85:100", SPN["wlmds_ratio"], round(r_, 2), round(r_, 2) == SPN["wlmds_ratio"],
      "SOURCED from WLMDS, not rebuildable from provider files")
check("P", "patients = ratio × pathways incl. estimates", round(pts), round(r_ * h.total_est), abs(pts - r_ * h.total_est) < 1,
      "consistency of NHS England's own arithmetic only")
check("P", "patients, to the notice's 0.1m", SPN["patients_rounded_m"], round(pts / 1e6, 1), round(pts / 1e6, 1) == SPN["patients_rounded_m"])

## Gate C — Cancer Waiting Times

My provider-basis sums against the release's percentages (to its one decimal
place) and against the national time series counts (exactly).

In [11]:
nts = openpyxl.load_workbook(CWT / "CWT-CRS-National-Time-Series-Oct-2009-Aug-2026-Provisional.xlsx",
                             read_only=True, data_only=True)["Monthly Performance"]
nts.reset_dimensions()
trows = list(nts.iter_rows(values_only=True))
# I locate each standard's block by its title text, then its columns by header
# text, because the sheet's layout has spacer columns I would rather not count.
TITLES = {"FDS": "Four Week (28 days)", "31D": "One Month (31-day)", "62D": "Two Month (62-day)"}
block = {m: next(i for i, x in enumerate(trows[1]) if x and str(x).startswith(t)) for m, t in TITLES.items()}
for m, c in block.items():
    assert (trows[3][c - 1], trows[3][c], trows[3][c + 1]) == ("Monthly", "Total", "Within Standard"), m
STANDARD_TEXT = {m: str(trows[2][c]) for m, c in block.items()}
NTS = {}
for r in trows[4:]:
    for m, c in block.items():
        if hasattr(r[c - 1], "strftime"):
            NTS[(m, r[c - 1].strftime("%Y-%m"))] = (r[c], r[c + 1])     # (Total, Within Standard)

cs = cwt.groupby(["measure", "period"])[["total", "within"]].sum()
for m in ["FDS", "31D", "62D"]:
    for p in ["2026-07", "2026-08"]:
        t, w = cs.loc[(m, p)]
        pct = round(100 * w / t, 1)
        check("C", f"{m} {p} % vs release", REL[f"{m}_{p}"], pct, pct == REL[f"{m}_{p}"])
        nt, nw = NTS[(m, p)]
        check("C", f"{m} {p} total & within vs national time series", f"{nt:,}/{nw:,}", f"{int(t):,}/{int(w):,}",
              (int(t), int(w)) == (int(nt), int(nw)))
check("C", "FDS Aug26 referrals seen vs release", REL["FDS_total_2026-08"], int(cs.loc[("FDS", "2026-08"), "total"]),
      int(cs.loc[("FDS", "2026-08"), "total"]) == REL["FDS_total_2026-08"])
check("C", "62D Aug26 treated vs release", REL["62D_total_2026-08"], int(cs.loc[("62D", "2026-08"), "total"]),
      int(cs.loc[("62D", "2026-08"), "total"]) == REL["62D_total_2026-08"])
STANDARD_TEXT

{'FDS': 'Operational Standard = 80% from Q1 2026/27',
 '31D': 'Operational Standard = 96% from Q3 2023/24',
 '62D': 'Operational Standard = 85% from Q3 2023/24'}

## The gate

In [12]:
gate = pd.DataFrame(checks)
gate.to_csv(OUT / "reconciliation.csv", index=False)
print(gate.groupby("gate")["pass"].agg(["sum", "count"]).rename(columns={"sum": "passed", "count": "checks"}))
gate

      passed  checks
gate                
A         21      21
A'         2       2
B          8       8
C         14      14
M          3       3
P          3       3


,gate,check,published,rebuilt,diff,pass,note
0,A,2026-08 incomplete: same provider set (zero-pathway listings allowed),533,532,-1.0,True,"1 workbook-only providers, all with 0 pathways; none missing from the workbook"
1,A,"2026-08 incomplete: total, provider by provider",7158164,7158164,0.0,True,all providers exact
2,A,"2026-08 incomplete: within_18, provider by provider",4667554,4667554,0.0,True,all providers exact
3,A,"2026-08 incomplete: over_52, provider by provider",109936,109936,0.0,True,all providers exact
4,A,"2026-08 incomplete: over_104, provider by provider",220,220,0.0,True,all providers exact
5,A,2026-08 incomplete_dta: same provider set (zero-pathway listings allowed),533,480,-53.0,True,"53 workbook-only providers, all with 0 pathways; none missing from the workbook"
6,A,"2026-08 incomplete_dta: total, provider by provider",1183326,1183326,0.0,True,all providers exact
7,A,"2026-08 incomplete_dta: over_104, provider by provider",61,61,0.0,True,all providers exact
8,A,2026-07 incomplete: same provider set (zero-pathway listings allowed),533,533,0.0,True,"0 workbook-only providers, all with 0 pathways; none missing from the workbook"
9,A,"2026-07 incomplete: total, provider by provider",7131685,7131685,0.0,True,all providers exact


## The claims, beside the primary figures

Each claim gets one of: **exact** (reconciles exactly), **rounding**
(reconciles to the precision quoted), **sourced** (true to the primary
source but not rebuildable from provider files), **does not reconcile**, or
**measure** (true only on the headline basis, not on my primary measure).

In [13]:
jul, aug = head.loc["2026-07"], head.loc["2026-08"]
cs62, csfds = cs.loc[("62D", "2026-08")], cs.loc[("FDS", "2026-08")]
pct = lambda a, b: 100 * (a / b - 1)
d52_h, d52_r = aug.over_52_est - jul.over_52_est, aug.over_52 - jul.over_52
p52_h, p52_r = pct(aug.over_52_est, jul.over_52_est), pct(aug.over_52, jul.over_52)
ptot_h, ptot_r = pct(aug.total_est, jul.total_est), pct(aug.total, jul.total)
med_aug, p62, pfds = med["2026-08"], 100 * cs62.within / cs62.total, 100 * csfds.within / csfds.total

def near(x, claim, places):
    return round(x, places) == round(claim, places)

# (headline-basis value, reported-basis value, verdict) - every verdict is computed from the numbers above.
primary = {
    "rtt_incomplete_pathways": (aug.total_est, aug.total,
        f"measure: {aug.total_est:,.0f} incl. estimates rounds to 7.37m {'(holds)' if near(aug.total_est / 1e6, 7.37, 2) else '(FAILS)'}; "
        f"my primary measure is {aug.total:,.0f}"),
    "rtt_unique_patients": (pts, None,
        f"sourced: WLMDS ratio {r_:.4f} × {aug.total_est:,.0f} = {pts:,.0f}; an estimate, not a count, not rebuildable"),
    "rtt_change_on_july": (aug.total_est - jul.total_est, aug.total - jul.total,
        f"measure: headline {aug.total_est - jul.total_est:+,.0f} {'(exact)' if aug.total_est - jul.total_est == 41_848 else '(FAILS)'}; "
        f"reported {aug.total - jul.total:+,.0f}"),
    "rtt_over_52_weeks": (aug.over_52_est, aug.over_52,
        f"measure: headline {aug.over_52_est:,.0f} {'(holds)' if near(aug.over_52_est / 1e3, 113, 0) else '(FAILS)'}; reported {aug.over_52:,.0f}"),
    "rtt_over_52_change": (d52_h, d52_r,
        f"{'holds' if abs(d52_h - 7_000) < 500 else 'does not reconcile'}: Jul->Aug {d52_h:+,.0f} headline / {d52_r:+,.0f} reported"),
    "rtt_over_52_pct_change": (p52_h, p52_r,
        f"{'holds' if abs(p52_h - 6) < 0.5 else 'does not reconcile'}: Jul->Aug {p52_h:+.1f}% headline / {p52_r:+.1f}% reported"),
    "rtt_total_pct_change": (ptot_h, ptot_r,
        f"{'holds' if ptot_h < 1 and ptot_r < 1 else 'does not reconcile'}: Jul->Aug {ptot_h:+.2f}% headline / {ptot_r:+.2f}% reported"),
    "rtt_median_weeks": (None, med_aug,
        f"{'rounding' if near(med_aug, 12.2, 1) else 'does not reconcile'}: {med_aug:.2f} weeks from bands (reported basis; NHS England publishes no headline-basis median)"),
    "cwt_62d_pct": (None, p62, f"{'rounding' if near(p62, 71.9, 1) else 'does not reconcile'}: {p62:.2f}%, provider basis, provisional"),
    "cwt_fds_pct": (None, pfds, f"{'rounding' if near(pfds, 78.6, 1) else 'does not reconcile'}: {pfds:.2f}%, provider basis, provisional"),
}
cv = pd.DataFrame([{"claim": k, "as_quoted": q, "claimed": v, "source": s,
                    "primary_headline_basis": None if primary[k][0] is None else round(float(primary[k][0]), 2),
                    "primary_reported_basis": None if primary[k][1] is None else round(float(primary[k][1]), 2),
                    "verdict": primary[k][2]} for k, v, _, q, s in CLAIMS])
cv.to_csv(OUT / "claims_vs_primary.csv", index=False)
cv[["as_quoted", "claimed", "primary_headline_basis", "primary_reported_basis", "verdict"]]

,as_quoted,claimed,primary_headline_basis,primary_reported_basis,verdict
0,7.37 million pathways,7370000.0,7370100.00,7158164.00,"measure: 7,370,100 incl. estimates rounds to 7.37m (holds); my primary measure is 7,15..."
1,6.24 million patients,6240000.0,6242270.83,NaN,"sourced: WLMDS ratio 0.8470 × 7,370,100 = 6,242,271; an estimate, not a count, not reb..."
2,"up 41,848 on July",41848.0,41848.00,26479.00,"measure: headline +41,848 (exact); reported +26,479"
3,"~113,000 waiting more than a year",113000.0,113339.00,109936.00,"measure: headline 113,339 (holds); reported 109,936"
4,"+7,000 over 52 weeks (article title)",7000.0,1960.00,1673.00,"does not reconcile: Jul->Aug +1,960 headline / +1,673 reported"
5,over-52-week group grew by roughly six per cent,6.0,1.76,1.55,does not reconcile: Jul->Aug +1.8% headline / +1.5% reported
6,list grew by well under one per cent,1.0,0.57,0.37,holds: Jul->Aug +0.57% headline / +0.37% reported
7,median wait 12.2 weeks,12.2,NaN,12.21,rounding: 12.21 weeks from bands (reported basis; NHS England publishes no headline-ba...
8,62-day performance 71.9 per cent,71.9,NaN,71.90,"rounding: 71.90%, provider basis, provisional"
9,28-day faster diagnosis 78.6 per cent,78.6,NaN,78.56,"rounding: 78.56%, provider basis, provisional"


In [14]:
# The gate itself. I raise rather than print, so a failure stops a scripted run.
failed = gate[~gate["pass"]]
if len(failed):
    print(failed.to_string())
    raise SystemExit(f"GATE: FAIL ({len(failed)} of {len(gate)} checks)")
print(f"GATE: PASS ({len(gate)} of {len(gate)} checks)")

GATE: PASS (51 of 51 checks)
